# Titanic Survival Analysis
## End-to-End EDA & Multivariate Analysis
**Dataset:** Kaggle Titanic Training Set (train.csv) — 891 passengers, real historical survival labels  
**Goal:** Predict survival and understand which factors drove it  
**Analytical question:** What combination of passenger characteristics best predicts survival?

---


## 1. Setup — install and import libraries

In [ ]:
# All libraries are pre-installed in Colab — no pip install needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family'] = 'sans-serif'
sns.set_palette('Set2')

print("✅ Libraries loaded")

## 2. Load data
Upload `train.csv` from Kaggle: https://www.kaggle.com/c/titanic/data  
Run the cell below then click the upload button that appears.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload train.csv here

import io
df_raw = pd.read_csv(io.BytesIO(uploaded['train.csv']))
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(10)

## 3. Raw data overview

In [ ]:
# Basic info
print("=== DATA TYPES ===")
print(df_raw.dtypes)
print("\n=== MISSING VALUES ===")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
print(pd.DataFrame({'missing': missing, 'pct': missing_pct})[missing > 0])
print("\n=== DESCRIPTIVE STATS ===")
df_raw.describe().round(2)

## 4. Missing data handling
**Strategy:**
- `Age` (20% missing) → impute with median by title group
- `Cabin` (77% missing) → binary flag `Has_cabin` only
- `Embarked` (2 missing) → fill with mode

In [ ]:
df = df_raw.copy()

# Extract title from name
df['Title'] = df['Name'].str.extract(r', ([A-Za-z]+)\.', expand=False)
df['Title'] = df['Title'].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Other')
df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

# Age: impute by title median
title_medians = df.groupby('Title')['Age'].median()
df['Age'] = df.apply(
    lambda r: title_medians[r['Title']] if pd.isna(r['Age']) else r['Age'], axis=1)

# Cabin: binary flag
df['Has_cabin'] = df['Cabin'].notna().astype(int)

# Embarked: fill mode
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

print("Missing values after imputation:")
print(df[['Age','Cabin','Embarked']].isnull().sum())
print(f"\nTitle distribution:")
print(df['Title'].value_counts())

## 5. Feature engineering
Creating new features that better capture the underlying patterns.

In [ ]:
# Family features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)
df['FamilyGroup'] = pd.cut(df['FamilySize'],
    bins=[0,1,3,5,20], labels=['Alone','Small','Medium','Large'])

# Fare transformation (log to handle skew)
df['LogFare'] = np.log1p(df['Fare'])

# Age features
df['IsChild']  = (df['Age'] < 10).astype(int)
df['AgeGroup'] = pd.cut(df['Age'],
    bins=[0,12,18,35,60,100], labels=['Child','Teen','YoungAdult','Adult','Senior'])

# Interaction feature
df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)

# Numeric encoding
df['Sex_num'] = (df['Sex'] == 'male').astype(int)

print("Engineered features added:")
new_cols = ['Title','FamilySize','IsAlone','FamilyGroup','LogFare','IsChild','AgeGroup','Sex_Pclass','Sex_num','Has_cabin']
print(new_cols)
df[new_cols].head()

## 6. EDA — survival rates by each factor

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

factors = [
    ('Sex',       'Sex'),
    ('Pclass',    'Passenger Class'),
    ('IsChild',   'Is Child (Age < 10)'),
    ('IsAlone',   'Traveling Alone'),
    ('FamilyGroup','Family Group'),
    ('AgeGroup',  'Age Group'),
]

overall = df['Survived'].mean()

for ax, (col, title) in zip(axes, factors):
    rates = df.groupby(col)['Survived'].mean().sort_values(ascending=False)
    colors = ['#1D9E75' if r >= overall else '#D85A30' for r in rates]
    bars = ax.bar(rates.index.astype(str), rates.values, color=colors, alpha=0.85, edgecolor='white')
    ax.axhline(overall, color='#888', linestyle='--', linewidth=1, label=f'Overall {overall:.1%}')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_ylabel('Survival rate')
    ax.set_ylim(0, 1.05)
    for bar, r in zip(bars, rates.values):
        ax.text(bar.get_x()+bar.get_width()/2, r+0.02, f'{r:.1%}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Survival rates by each factor\n(teal = above average, red = below average)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Sex × Class interaction heatmap — the most important multivariate view

In [ ]:
pivot = df.pivot_table('Survived', index='Sex', columns='Pclass', aggfunc='mean')
counts = df.pivot_table('Survived', index='Sex', columns='Pclass', aggfunc='count')

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=False, fmt='.1%', cmap='RdYlGn',
            vmin=0, vmax=1, ax=ax, linewidths=0.5)

# Add custom annotations with rate + count
for i, sex in enumerate(pivot.index):
    for j, cls in enumerate(pivot.columns):
        rate = pivot.loc[sex, cls]
        n    = int(counts.loc[sex, cls])
        ax.text(j+0.5, i+0.4, f'{rate:.1%}',
                ha='center', va='center', fontsize=14, fontweight='bold',
                color='white' if rate > 0.6 or rate < 0.2 else 'black')
        ax.text(j+0.5, i+0.65, f'n={n}',
                ha='center', va='center', fontsize=10, color='white' if rate > 0.6 or rate < 0.2 else 'black')

ax.set_title('Survival rate by Sex × Passenger Class\n(real historical rates)', 
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Passenger Class', fontsize=11)
ax.set_ylabel('Sex', fontsize=11)
plt.tight_layout()
plt.show()

print("\nKey insight: 3rd class women survived at only 50% — class mattered enormously for women.")
print("1st class men survived at 37% — wealth gave men a real but modest advantage.")

## 8. Kernel Density Estimation (KDE)
Comparing the full distribution shape of Age and Fare between survivors and non-survivors.  
**KDE = smooth histogram built by placing a bell curve on each data point and summing.**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

surv  = df[df['Survived']==1]
nsurv = df[df['Survived']==0]

# Age KDE
ax = axes[0]
surv['Age'].plot.kde(ax=ax, label='Survived', color='#1D9E75', linewidth=2.5)
nsurv['Age'].plot.kde(ax=ax, label='Did not survive', color='#B4B2A9', linewidth=2.5)
ax.fill_between(np.linspace(0,80,200),
    gaussian_kde(surv['Age'])(np.linspace(0,80,200)),
    alpha=0.15, color='#1D9E75')
ax.fill_between(np.linspace(0,80,200),
    gaussian_kde(nsurv['Age'])(np.linspace(0,80,200)),
    alpha=0.12, color='#B4B2A9')
ax.axvline(10, color='#D85A30', linestyle=':', linewidth=1.5, label='IsChild threshold (age 10)')
ax.set_xlim(0, 80)
ax.set_title('Age KDE by survival outcome', fontsize=12, fontweight='bold')
ax.set_xlabel('Age'); ax.set_ylabel('Density')
ax.legend(fontsize=10)
ax.text(3, ax.get_ylim()[1]*0.85, 'Child\nsurvival\nbump', fontsize=9, color='#D85A30', ha='center')

# LogFare KDE
ax = axes[1]
surv['LogFare'].plot.kde(ax=ax, label='Survived', color='#1D9E75', linewidth=2.5)
nsurv['LogFare'].plot.kde(ax=ax, label='Did not survive', color='#B4B2A9', linewidth=2.5)
ax.fill_between(np.linspace(0,7,200),
    gaussian_kde(surv['LogFare'])(np.linspace(0,7,200)),
    alpha=0.15, color='#1D9E75')
ax.fill_between(np.linspace(0,7,200),
    gaussian_kde(nsurv['LogFare'])(np.linspace(0,7,200)),
    alpha=0.12, color='#B4B2A9')
ax.set_xlim(0, 7)
ax.set_title('log(Fare+1) KDE by survival outcome', fontsize=12, fontweight='bold')
ax.set_xlabel('log(Fare + 1)'); ax.set_ylabel('Density')
ax.legend(fontsize=10)

plt.suptitle('KDE: where distributions separate tells you what to model',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nTherefore:")
print("  Fare → keep, log-transform confirmed, engineer FareBand")
print("  Age  → keep, engineer IsChild flag (age < 10), use raw age as secondary feature")

## 9. Correlation matrix — multicollinearity detection

In [ ]:
model_cols = ['Survived','Sex_num','Pclass','LogFare','Age','FamilySize','IsAlone','IsChild','SibSp','Parch']
col_labels = ['Survived','Sex','Pclass','LogFare','Age','FamilySize','IsAlone','IsChild','SibSp','Parch']

corr = df[model_cols].corr()
corr.index   = col_labels
corr.columns = col_labels

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Full correlation heatmap
mask = np.zeros_like(corr, dtype=bool)  # show full matrix
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, ax=axes[0], linewidths=0.5,
            annot_kws={'size': 9}, square=True)
axes[0].set_title('Pearson Correlation Matrix\n(all features)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Correlation with survival only
surv_corr = corr['Survived'].drop('Survived').sort_values(key=abs, ascending=True)
colors = ['#1D9E75' if v > 0 else '#D85A30' for v in surv_corr]
axes[1].barh(surv_corr.index, surv_corr.values, color=colors, alpha=0.85, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].axvline(0.3,  color='#1D9E75', linestyle='--', linewidth=1, alpha=0.5, label='|r|=0.3 threshold')
axes[1].axvline(-0.3, color='#1D9E75', linestyle='--', linewidth=1, alpha=0.5)
for i, (v, name) in enumerate(zip(surv_corr.values, surv_corr.index)):
    axes[1].text(v + (0.01 if v >= 0 else -0.01), i,
                 f'{v:+.3f}', va='center', ha='left' if v >= 0 else 'right', fontsize=10)
axes[1].set_title('Correlation with Survival\n(ranked by strength)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Pearson r')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\n=== HIGH CORRELATIONS (|r| > 0.5) — MULTICOLLINEARITY FLAGS ===")
for i in range(len(model_cols)):
    for j in range(i+1, len(model_cols)):
        r = corr.iloc[i,j]
        if abs(r) > 0.5:
            print(f"  {col_labels[i]:12s} × {col_labels[j]:12s}: r={r:+.3f}  → {'DROP one' if abs(r)>0.7 else 'WATCH'}")

## 10. Covariance matrix
Covariance = correlation in original units. Scale-dependent — Age dominates because σ≈13.  
**Rule:** Use correlation for interpretation. Use covariance for PCA and distance-based algorithms.

In [ ]:
clean_cols   = ['Sex_num','Pclass','LogFare','Age','IsAlone','IsChild']
clean_labels = ['Sex','Pclass','LogFare','Age','IsAlone','IsChild']

cov = df[clean_cols].cov()
cov.index   = clean_labels
cov.columns = clean_labels

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Covariance heatmap
sns.heatmap(cov, annot=True, fmt='.3f', cmap='RdYlGn',
            ax=axes[0], linewidths=0.5, annot_kws={'size':9}, square=True)
axes[0].set_title('Covariance Matrix\n(clean feature set)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Side-by-side: cov vs corr for Age x LogFare
pairs = [('Age × LogFare', 1.40, 0.111),
         ('Age × IsChild', -1.761, -0.531),
         ('IsAlone × LogFare', -0.227, -0.478),
         ('Pclass × LogFare', -0.536, -0.661)]
x = np.arange(len(pairs))
w = 0.35
labels_p = [p[0] for p in pairs]
covs = [p[1] for p in pairs]
corrs = [p[2] for p in pairs]

ax2 = axes[1]
ax2.bar(x - w/2, [abs(c) for c in covs],  w, label='|Covariance|', color='#378ADD', alpha=0.8)
ax2.bar(x + w/2, [abs(c) for c in corrs], w, label='|Correlation|', color='#1D9E75', alpha=0.8)
ax2.set_xticks(x); ax2.set_xticklabels(labels_p, rotation=15, ha='right', fontsize=9)
ax2.set_title('Covariance vs Correlation magnitude\n(same pairs, different scales)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Absolute value')
ax2.legend(fontsize=10)
ax2.text(0.02, 0.95, 'Age dominates covariance\nbecause σ_Age ≈ 13',
         transform=ax2.transAxes, fontsize=9, color='#D85A30',
         va='top', bbox=dict(boxstyle='round', facecolor='#FAECE7', alpha=0.8))

plt.tight_layout()
plt.show()

## 11. Scatter plots — each variable vs survival

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
surv  = df[df['Survived']==1]
nsurv = df[df['Survived']==0]

np.random.seed(42)
plots = [
    ('Age',      'Age',            0.3),
    ('LogFare',  'log(Fare+1)',     0.08),
    ('FamilySize','Family size',   0.2),
    ('Pclass',   'Passenger class',0.15),
]

for ax, (col, xlabel, jitter) in zip(axes, plots):
    js = lambda n: np.random.uniform(-jitter, jitter, n)
    ax.scatter(surv[col]  + js(len(surv)),  np.ones(len(surv))*0.7  + js(len(surv)),
               alpha=0.4, color='#1D9E75', s=25, label='Survived')
    ax.scatter(nsurv[col] + js(len(nsurv)), np.ones(len(nsurv))*0.3 + js(len(nsurv)),
               alpha=0.3, color='#B4B2A9', s=20, label='Did not survive')
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_yticks([0.3, 0.7])
    ax.set_yticklabels(['Did not\nsurvive', 'Survived'], fontsize=9)
    r, _ = stats.pointbiserialr(df['Survived'], df[col])
    ax.set_title(f'{xlabel} vs Survival   r = {r:+.3f}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right')
    ax.set_ylim(0, 1)

plt.suptitle('Jittered scatter — each variable vs survival outcome',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 12. Clean feature set for modeling
Based on correlation analysis, KDE, and multicollinearity detection.

In [ ]:
# Final feature selection decisions
print("=== FEATURE SELECTION DECISIONS ===")
print()
print("KEEP:")
keep = {
    'Sex':      'r=-0.543 (strongest predictor)',
    'Pclass':   'r=-0.338 (class signal)',  
    'LogFare':  'r=+0.330 (wealth signal, log-transformed)',
    'IsAlone':  'r=-0.203 (family structure)',
    'IsChild':  'r=+0.129 (captures child survival bump in KDE)',
    'Age':      'r=-0.065 (weak but non-linear — retain as secondary)',
}
for f, reason in keep.items():
    print(f"  ✅ {f:12s} {reason}")

print()
print("DROP (multicollinearity):")
drop = {
    'FamilySize': 'r=+0.891 with SibSp — IsAlone captures this better',
    'SibSp':      'r=+0.891 with FamilySize — redundant',
    'Parch':      'r=+0.783 with FamilySize — redundant',
}
for f, reason in drop.items():
    print(f"  ❌ {f:12s} {reason}")

# Build model-ready dataframe
model_features = ['Survived','Sex_num','Pclass','LogFare','Age','IsAlone','IsChild']
df_model = df[model_features].copy()
df_model.columns = ['Survived','Sex','Pclass','LogFare','Age','IsAlone','IsChild']
print(f"\nModel-ready dataset: {df_model.shape}")
print(df_model.describe().round(3))

## 13. What's next

### Analysis completed ✅
1. EDA — survival rates across all factors
2. Missing data — imputation by title group
3. Feature engineering — Title, IsChild, IsAlone, LogFare, FamilySize
4. Distribution analysis — KDE, normality tests, transformations
5. Correlation matrix — multicollinearity detection
6. Covariance matrix — scale effects explained
7. Scatter plots — visual separation by variable

### Next steps ⬜
- **Joint distribution** — 2D KDE for pairs of variables
- **Copula analysis** — dependency structure beyond correlation
- **PCA** — dimensionality reduction, visualize variance
- **Model building** — Logistic Regression → Random Forest → XGBoost
- **Model evaluation** — Confusion matrix, AUC-ROC, feature importance
- **Portfolio artifacts** — Case study PDF, GitHub README

### Director-level note
The clean feature set is: `Sex, Pclass, LogFare, Age, IsAlone, IsChild`  
Before modeling, verify these decisions hold in cross-validation.  
The Sex × Pclass interaction is the single most powerful combined signal — consider it as an explicit feature.
